# Imports

In [558]:
import os

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, pointbiserialr
from scipy.stats import f_oneway, kruskal

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, roc_curve, auc
from sklearn.isotonic import IsotonicRegression

from autogluon.tabular import TabularDataset, TabularPredictor
#from autogluon.multimodal import MultiModalPredictor
import xgboost as xgb

plt.style.use('default')
sns.set_palette("husl")
SEED = 43
np.random.seed(SEED)

VAL_TILE_FRACTION = 0.4
TILE_SIZE_DEG = 0.01  # approx 1km


SCORING = {
    'r2': 'r2',
    'mae': 'neg_mean_absolute_error',
    'mse': 'neg_mean_squared_error'
}
GRID_SEARCH_KWARGS = dict(
    cv=10,
    scoring=SCORING,
    refit='r2',
    n_jobs=-1,
    return_train_score=True
)
LINEAR_PARAM_GRID = {
    'model__fit_intercept': [True, False],
    'model__positive': [False, True]
}
TREE_PARAM_GRID = {
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [1, 100],
    'model__min_samples_leaf': [2, 100],
    'model__max_features': [None, 'sqrt', 'log2']
}
RF_PARAM_GRID = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [1, 100],
    'model__min_samples_leaf': [2, 100],
    'model__max_features': [None, 'sqrt', 'log2']
}
MLP_PARAM_GRID = {
    'model__hidden_layer_sizes': [(100,), (100, 50), (50, 25)],
    'model__activation': ['relu', 'tanh'],
    'model__alpha': [0.0001, 0.001, 0.01],
    'model__learning_rate_init': [0.001, 0.01]
}
XGB_PARAM_GRID = {
    'model__learning_rate': [0.01, 0.1, 0.2],
    'model__n_estimators': [50, 100, 200]
}

def evaluate_regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    errors = y_true - y_pred

    bin_size = 0.1
    bins = np.arange(0.0, 1.0 + bin_size, bin_size)
    bin_ids = np.digitize(y_pred, bins, right=False)

    ece = 0.0
    N = len(y_pred)

    for i in range(1, len(bins)):
        mask = bin_ids == i
        if not np.any(mask):
            continue

        bin_center = (bins[i - 1] + bins[i]) / 2.0
        median_obs = np.median(y_true[mask])
        ece += (mask.sum() / N) * abs(median_obs - bin_center)
        
    return {
        'r2': r2_score(y_true, y_pred),
        'mae': float(np.mean(np.abs(errors))),
        'mse': float(np.mean(errors ** 2)),
        "p95": np.quantile(np.abs(errors), 0.95),
        'ece': float(ece),              # Expected calibration error
    }


def record_regression_results(results, target, model_name, y_true, y_pred, split='test'):
    metrics = evaluate_regression_metrics(y_true, y_pred)
    split_label = split.capitalize()
    print(f"{split_label} R2 score {metrics['r2']:.4f}")
    print(f"{split_label} MAE score {metrics['mae']:.4f}")
    print(f"{split_label} MSE score {metrics['mse']:.4f}")
    print(f"{split_label} P95 score {metrics['p95']:.4f}")
    print(f"{split_label} ECE score {metrics['ece']:.4f}")
    results.append({
        'target': target,
        'model': model_name,
        f'{split}_r2': metrics['r2'],
        f'{split}_mae': metrics['mae'],
        #f'{split}_mse': metrics['mse'],
        #f'{split}_p95': metrics['p95'],
        f'{split}_ece': metrics['ece'],
    })
    return metrics


# Load data

In [559]:
model_det = "faster-rcnn"
data_path_file = f"{model_det}_metafeatures.csv"
data = pd.read_csv("../data/" + data_path_file, index_col=0)

print(data)

      country time_of_day        lat       long       road_type  \
1          PL         day  52.249511  21.043137            city   
6          PL         day  52.239332  21.030383            city   
39         PL         day  52.234241  21.003985            city   
49         PL         day  52.224666  21.071192            city   
52         PL    twilight  52.231355  21.054809            city   
...       ...         ...        ...        ...             ...   
99904      PL         day  54.353453  18.645450  arterial-urban   
99939      DE       night  50.116028   8.622851         highway   
99941      DE         day  53.057472   8.958626  arterial-urban   
99984      IT       night  41.935093  12.599486   smaller-rural   
99997      SE    twilight  59.103593  12.789612  arterial-rural   

      road_condition              weather  solar_angle_elevation  month  hour  \
1             normal    partly-cloudy-day              51.723833      4    10   
6             normal            c

In [560]:
data = data.dropna()
num_rows, num_cols = data.shape
print(f"After dropping missing values: {num_rows:,} rows and {num_cols} columns")

After dropping missing values: 9,655 rows and 42 columns


In [561]:
iou = data["iou"]
lrp = data["lrp"]
conf = data["mean_conf"]
image_paths_dict = {int(img_pth.split("_")[0]):f"../data/zod_yolo/images/test/{img_pth}" for img_pth in os.listdir("../data/zod_yolo/images/test") if img_pth.endswith(".jpg")}
img_path = pd.DataFrame.from_dict(image_paths_dict, orient='index', columns=["Images"])

data = data.drop(columns=["iou", "lrp"])
data_indices = data.index.to_numpy()

In [562]:
all_columns = data.columns    
all_columns = all_columns.drop(["weather_code", "cloud_cover_low", "cloud_cover_mid"])
all_columns = all_columns.drop(['temperature_2m', 'relative_humidity_2m', 'rain', 'snowfall', 'cloud_cover', 'wind_speed_10m'])
all_columns = all_columns.drop(["road_type", "road_condition", "weather"])

data = data[all_columns]
print(f"Number of rows: {data.shape[0]}")
print(f"Columns: {all_columns.sort_values()}")
print(f"Number of columns: {len(all_columns)}")

Number of rows: 9655
Columns: Index(['brightness', 'camera_distance_from_ground', 'camera_offset',
       'camera_pitch_angle', 'complexity', 'contrast', 'country',
       'distortion_magnitude', 'field_view_horizontal', 'forward_acceleration',
       'forward_velocity', 'hour', 'laplacian', 'lat', 'lateral_acceleration',
       'lateral_velocity', 'long', 'max_conf', 'mean_conf', 'min_conf',
       'month', 'noisiness', 'num_detections', 'quality', 'sharpness',
       'solar_angle_elevation', 'std_conf', 'time_of_day'],
      dtype='object')
Number of columns: 28


In [563]:
numeric_columns = ['solar_angle_elevation', 'forward_acceleration', 'lateral_acceleration', 'forward_velocity', 'lateral_velocity', 'field_view_horizontal', 'camera_distance_from_ground', 'camera_pitch_angle', 'distortion_magnitude', 'camera_offset', 'laplacian', 'brightness', 'contrast', 'sharpness', 'noisiness', 'quality', 'complexity']
numeric_columns = numeric_columns + ['mean_conf', "std_conf", "max_conf", "min_conf", "num_detections"]

categorical_columns = ["country", "time_of_day"]
other = ["hour", "month", "lat", "long"]
print(sorted(numeric_columns + categorical_columns + other))
assert len(all_columns) == (len(categorical_columns) + len(numeric_columns) + len(other)), "Columns not match"

['brightness', 'camera_distance_from_ground', 'camera_offset', 'camera_pitch_angle', 'complexity', 'contrast', 'country', 'distortion_magnitude', 'field_view_horizontal', 'forward_acceleration', 'forward_velocity', 'hour', 'laplacian', 'lat', 'lateral_acceleration', 'lateral_velocity', 'long', 'max_conf', 'mean_conf', 'min_conf', 'month', 'noisiness', 'num_detections', 'quality', 'sharpness', 'solar_angle_elevation', 'std_conf', 'time_of_day']


# Assessor Tests

Split data into train 60% and validation 40%. To prevent leakage, the data is binned in buckets by lat/long in 1km. The split is done by buckets ensuring no leakage. 

In [564]:
#Val split
def tile_id(lat, lon, size_deg):
    lat_bin = np.floor(lat / size_deg) * size_deg
    lon_bin = np.floor(lon / size_deg) * size_deg
    return f'{lat_bin:.4f}_{lon_bin:.4f}'

In [565]:
## Make split of train into train and val by tiles
ids = data.index.tolist()
coords = {frame_id: {"lat": data.loc[frame_id, "lat"], "lon": data.loc[frame_id, "long"]} for frame_id in ids}
df_coords = pd.DataFrame.from_dict(coords, orient='index')
df_coords['tile_id'] = [tile_id(lat, lon, TILE_SIZE_DEG) for lat, lon in zip(df_coords['lat'], df_coords['lon'])]

unique_tiles = sorted(set(df_coords['tile_id'].dropna().unique()))
rng = np.random.default_rng(SEED)
val_tile_count = int(len(unique_tiles) * VAL_TILE_FRACTION)
val_tiles = set(rng.choice(unique_tiles, size=val_tile_count, replace=False))
df_coords['split'] = np.where(df_coords['tile_id'].isin(val_tiles), 'val', 'train')
test_idx = df_coords[df_coords['split'] == 'val'].index.tolist()
train_idx = df_coords[df_coords['split'] == 'train'].index.tolist()
print(f"Total train frames: {len(train_idx)}, val frames: {len(test_idx)}")

Total train frames: 5518, val frames: 4137


In [566]:
X_train = data.loc[train_idx]
y_train = iou.loc[train_idx]
y_train_lrp = lrp.loc[train_idx]
conf_train = conf.loc[train_idx]
conf_train = pd.DataFrame(conf_train)

X_test = data.loc[test_idx]
y_test = iou.loc[test_idx]
y_test_lrp = lrp.loc[test_idx]
conf_test = conf.loc[test_idx]
conf_test = pd.DataFrame(conf_test)

print("X:", len(X_train), len(X_test))
print("y:", len(y_train), len(y_test))
print("Conf:", len(conf_train), len(conf_test))
print(X_train.columns)

X: 5518 4137
y: 5518 4137
Conf: 5518 4137
Index(['country', 'time_of_day', 'lat', 'long', 'solar_angle_elevation',
       'month', 'hour', 'forward_acceleration', 'lateral_acceleration',
       'forward_velocity', 'lateral_velocity', 'field_view_horizontal',
       'camera_distance_from_ground', 'camera_pitch_angle',
       'distortion_magnitude', 'camera_offset', 'laplacian', 'quality',
       'brightness', 'noisiness', 'sharpness', 'contrast', 'complexity',
       'num_detections', 'max_conf', 'min_conf', 'mean_conf', 'std_conf'],
      dtype='object')


In [567]:
model_results = []

### IoU

#### Baseline

Random Prediction. Mean of metric over training set.

In [568]:
mean_iou_train = np.mean(y_train)
iou_pred_test = np.full_like(y_test, mean_iou_train)
record_regression_results(model_results, 'IoU', 'Constant Mean Predictor', y_test, iou_pred_test)


Test R2 score -0.0063
Test MAE score 0.1472
Test MSE score 0.0362
Test P95 score 0.4089
Test ECE score 0.0134


{'r2': -0.0063339086884728335,
 'mae': 0.14724759906224852,
 'mse': 0.036165032413034294,
 'p95': 0.4088632476996413,
 'ece': 0.0134445172089796}

#### Linear Reg on Conf

In [569]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['mean_conf']),
    ],
    remainder='passthrough'
)
linear_reg_conf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
linear_reg_conf_grid = GridSearchCV(
    linear_reg_conf_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_conf_grid.fit(conf_train, y_train)

best_idx = linear_reg_conf_grid.best_index_
mean_train_r2 = linear_reg_conf_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = linear_reg_conf_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -linear_reg_conf_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -linear_reg_conf_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -linear_reg_conf_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -linear_reg_conf_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {linear_reg_conf_grid.best_params_}")
best_linear_reg_conf = linear_reg_conf_grid.best_estimator_


Mean CV train r2 score 0.3291
Mean CV test r2 score 0.2406
Mean CV train MAE score 0.1215
Mean CV test MAE score 0.1228
Mean CV train MSE score 0.0240
Mean CV test MSE score 0.0245
Best params: {'model__fit_intercept': True, 'model__positive': False}


In [570]:
y_pred_iou_baseline = best_linear_reg_conf.predict(conf_test)
record_regression_results(model_results, 'IoU', 'Univariate Linear Regression (Confidence)', y_test, y_pred_iou_baseline)


Test R2 score 0.3245
Test MAE score 0.1214
Test MSE score 0.0243
Test P95 score 0.3130
Test ECE score 0.0391


{'r2': 0.32454708903345897,
 'mae': 0.1214199442151586,
 'mse': 0.02427402694839069,
 'p95': 0.31297186622648476,
 'ece': 0.03908711677965764}

#### Isotonic Regression on Conf


In [571]:
iso_iou = IsotonicRegression(out_of_bounds="clip")
iso_iou.fit(conf_train["mean_conf"].to_numpy(), y_train.to_numpy())
y_pred_iou_iso = iso_iou.predict(conf_test["mean_conf"].to_numpy())
record_regression_results(model_results, 'IoU', 'Isotonic Regression (Confidence)', y_test, y_pred_iou_iso)


Test R2 score 0.4291
Test MAE score 0.1083
Test MSE score 0.0205
Test P95 score 0.2783
Test ECE score 0.0093


{'r2': 0.42912986670358844,
 'mae': 0.10826690962001112,
 'mse': 0.020515592981662212,
 'p95': 0.2782758533669687,
 'ece': 0.00934257020614187}

#### Linear Regression

Train models with cv and then test.

In [572]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
linear_reg_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
linear_reg_grid = GridSearchCV(
    linear_reg_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_grid.fit(X_train, y_train)

best_idx = linear_reg_grid.best_index_
mean_train_r2 = linear_reg_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = linear_reg_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -linear_reg_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -linear_reg_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -linear_reg_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -linear_reg_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {linear_reg_grid.best_params_}")
best_linear_reg = linear_reg_grid.best_estimator_

Mean CV train r2 score 0.4510
Mean CV test r2 score 0.3692
Mean CV train MAE score 0.1088
Mean CV test MAE score 0.1103
Mean CV train MSE score 0.0197
Mean CV test MSE score 0.0202
Best params: {'model__fit_intercept': True, 'model__positive': False}


In [573]:
y_pred_iou_lr = best_linear_reg.predict(X_test)
record_regression_results(model_results, 'IoU', 'Linear Regression', y_test, y_pred_iou_lr)


Test R2 score 0.4462
Test MAE score 0.1078
Test MSE score 0.0199
Test P95 score 0.2828
Test ECE score 0.0166


{'r2': 0.4461801085556737,
 'mae': 0.10783750359738598,
 'mse': 0.019902851481145387,
 'p95': 0.28276798667397446,
 'ece': 0.016595036618143592}

#### Decision Trees

In [574]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
decision_tree_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=SEED))
])
decision_tree_grid = GridSearchCV(
    decision_tree_pipeline,
    param_grid=TREE_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
decision_tree_grid.fit(X_train, y_train)

best_idx = decision_tree_grid.best_index_
mean_train_r2 = decision_tree_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = decision_tree_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -decision_tree_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -decision_tree_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -decision_tree_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -decision_tree_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {decision_tree_grid.best_params_}")
best_decision_tree = decision_tree_grid.best_estimator_


Mean CV train r2 score 0.4874
Mean CV test r2 score 0.3727
Mean CV train MAE score 0.1037
Mean CV test MAE score 0.1090
Mean CV train MSE score 0.0184
Mean CV test MSE score 0.0202
Best params: {'model__max_depth': None, 'model__max_features': None, 'model__min_samples_leaf': 100, 'model__min_samples_split': 100}


/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
180 fits failed out of a total of 360.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
180 fits failed with the following error:
Traceback (most recent call last):
  File "/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/felix/miniconda3/envs/pred_ad/lib/py

In [575]:
y_pred_iou_dt = best_decision_tree.predict(X_test)
record_regression_results(model_results, 'IoU', 'Decision Tree', y_test, y_pred_iou_dt)


Test R2 score 0.4417
Test MAE score 0.1069
Test MSE score 0.0201
Test P95 score 0.2832
Test ECE score 0.0179


{'r2': 0.4416570379074305,
 'mae': 0.10687408666923182,
 'mse': 0.020065398917128482,
 'p95': 0.2831634131931292,
 'ece': 0.017859854061159246}

#### Random Forest

In [576]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
rand_forest_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=SEED, n_jobs=-1))
])
rand_forest_grid = GridSearchCV(
    rand_forest_pipeline,
    param_grid=RF_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
rand_forest_grid.fit(X_train, y_train)

best_idx = rand_forest_grid.best_index_
mean_train_r2 = rand_forest_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = rand_forest_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -rand_forest_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -rand_forest_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -rand_forest_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -rand_forest_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {rand_forest_grid.best_params_}")
best_rand_forest = rand_forest_grid.best_estimator_


/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
360 fits failed out of a total of 720.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
360 fits failed with the following error:
Traceback (most recent call last):
  File "/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/felix/miniconda3/envs/pred_ad/lib/py

Mean CV train r2 score 0.5936
Mean CV test r2 score 0.4050
Mean CV train MAE score 0.0926
Mean CV test MAE score 0.1055
Mean CV train MSE score 0.0146
Mean CV test MSE score 0.0190
Best params: {'model__max_depth': 20, 'model__max_features': None, 'model__min_samples_leaf': 2, 'model__min_samples_split': 100, 'model__n_estimators': 200}


In [577]:
y_pred_iou_rf = best_rand_forest.predict(X_test)
record_regression_results(model_results, 'IoU', 'Random Forest', y_test, y_pred_iou_rf)


Test R2 score 0.4759
Test MAE score 0.1033
Test MSE score 0.0188
Test P95 score 0.2762
Test ECE score 0.0134


{'r2': 0.47593675980824646,
 'mae': 0.10327963157140706,
 'mse': 0.01883347455986568,
 'p95': 0.2762254217434038,
 'ece': 0.013352095430824369}

#### MLP

In [578]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
mlp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', MLPRegressor(max_iter=500, random_state=SEED))
])
mlp_grid = GridSearchCV(
    mlp_pipeline,
    param_grid=MLP_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
mlp_grid.fit(X_train, y_train)

best_idx = mlp_grid.best_index_
mean_train_r2 = mlp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = mlp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -mlp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -mlp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -mlp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -mlp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {mlp_grid.best_params_}")
best_mlp = mlp_grid.best_estimator_


Mean CV train r2 score 0.5047
Mean CV test r2 score 0.3838
Mean CV train MAE score 0.1029
Mean CV test MAE score 0.1083
Mean CV train MSE score 0.0177
Mean CV test MSE score 0.0198
Best params: {'model__activation': 'tanh', 'model__alpha': 0.01, 'model__hidden_layer_sizes': (100, 50), 'model__learning_rate_init': 0.01}


In [579]:
y_pred_iou_mlp = best_mlp.predict(X_test)
record_regression_results(model_results, 'IoU', 'MLP', y_test, y_pred_iou_mlp)


Test R2 score 0.4679
Test MAE score 0.1039
Test MSE score 0.0191
Test P95 score 0.2820
Test ECE score 0.0104


{'r2': 0.4679341163414378,
 'mae': 0.1039116480248997,
 'mse': 0.019121068824421748,
 'p95': 0.2820221917319501,
 'ece': 0.010388205110203727}

#### XGBoost

In [580]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb.XGBRegressor())
])
xgb_grid = GridSearchCV(
    xgb_pipeline,
    param_grid=XGB_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
xgb_grid.fit(X_train, y_train)


best_idx = xgb_grid.best_index_
mean_train_r2 = xgb_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = xgb_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -xgb_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -xgb_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -xgb_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -xgb_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {xgb_grid.best_params_}")
best_xgb = xgb_grid.best_estimator_

Mean CV train r2 score 0.7371
Mean CV test r2 score 0.3936
Mean CV train MAE score 0.0765
Mean CV test MAE score 0.1061
Mean CV train MSE score 0.0094
Mean CV test MSE score 0.0194
Best params: {'model__learning_rate': 0.1, 'model__n_estimators': 50}


In [581]:
y_pred_iou_xgb = best_xgb.predict(X_test)
record_regression_results(model_results, 'IoU', 'XGBoost', y_test, y_pred_iou_xgb)


Test R2 score 0.4664
Test MAE score 0.1038
Test MSE score 0.0192
Test P95 score 0.2866
Test ECE score 0.0116


{'r2': 0.4664172031348631,
 'mae': 0.10379291618663136,
 'mse': 0.019175582753456523,
 'p95': 0.2866072680268968,
 'ece': 0.011631038129694125}

#### Autogluon

##### Tabluar

In [582]:
train = pd.merge(X_train, y_train, left_index=True, right_index=True)

In [583]:
autoglue_model = TabularPredictor(label="iou", problem_type="regression", eval_metric="r2", path="../models/assessors/autoglue_tab/iou/")
autoglue_predictor = autoglue_model.fit(train)


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.14
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #100-Ubuntu SMP PREEMPT_DYNAMIC Tue Jan 13 16:40:06 UTC 2026
CPU Count:          32
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 23.55/23.55 GB
Total GPU Memory:   Free: 23.55 GB, Allocated: 0.00 GB, Total: 23.55 GB
GPU Count:          1
Memory Avail:       4.30 GB / 30.46 GB (14.1%)
Disk Space Avail:   1586.98 GB / 3542.28 GB (44.8%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : New in v1.5: The state-of-the-art for tabular data. Massively better than 'best' on datasets <100000 samples by using new 

Beginning AutoGluon training ...
AutoGluon will save models to "/home/felix/Predictability-AD/models/assessors/autoglue_tab/iou"
Train Data Rows:    5518
Train Data Columns: 28
Label Column:       iou
Problem Type:       regression
Preprocessing data ...
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    4394.93 MB
	Train Data (Original)  Memory Usage: 1.72 MB (0.0% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fitting CategoryMemoryMinimizeFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	T

In [584]:
autoglue_predictor.fit_summary(verbosity=2)

*** Summary of fit() ***
Estimated performance of each model:
                 model  score_val eval_metric  pred_time_val  fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0  WeightedEnsemble_L2   0.506326          r2       0.044061  5.660711                0.000212           0.015429            2       True          9
1           LightGBMXT   0.501812          r2       0.001789  1.335840                0.001789           1.335840            1       True          1
2              XGBoost   0.490882          r2       0.011575  0.387096                0.011575           0.387096            1       True          6
3             LightGBM   0.486614          r2       0.000979  0.396566                0.000979           0.396566            1       True          2
4             CatBoost   0.481919          r2       0.001512  0.729880                0.001512           0.729880            1       True          4
5        LightGBMLarge   0.481862          r

/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/autogluon/core/utils/plots.py:169: UserWarning: AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"
  warnings.warn('AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"')


{'model_types': {'LightGBMXT': 'LGBModel',
  'LightGBM': 'LGBModel',
  'RandomForestMSE': 'RFModel',
  'CatBoost': 'CatBoostModel',
  'ExtraTreesMSE': 'XTModel',
  'XGBoost': 'XGBoostModel',
  'NeuralNetTorch': 'TabularNeuralNetTorchModel',
  'LightGBMLarge': 'LGBModel',
  'WeightedEnsemble_L2': 'WeightedEnsembleModel'},
 'model_performance': {'LightGBMXT': 0.5018119224787093,
  'LightGBM': 0.48661444473298865,
  'RandomForestMSE': 0.481767390730784,
  'CatBoost': 0.4819192742768357,
  'ExtraTreesMSE': 0.48148555876842236,
  'XGBoost': 0.49088189740791965,
  'NeuralNetTorch': 0.46707357333532207,
  'LightGBMLarge': 0.481862303981921,
  'WeightedEnsemble_L2': 0.5063263441037312},
 'model_best': 'WeightedEnsemble_L2',
 'model_paths': {'LightGBMXT': ['LightGBMXT'],
  'LightGBM': ['LightGBM'],
  'RandomForestMSE': ['RandomForestMSE'],
  'CatBoost': ['CatBoost'],
  'ExtraTreesMSE': ['ExtraTreesMSE'],
  'XGBoost': ['XGBoost'],
  'NeuralNetTorch': ['NeuralNetTorch'],
  'LightGBMLarge': ['Ligh

In [585]:
y_pred_iou_autg = autoglue_predictor.predict(X_test)
record_regression_results(model_results, 'IoU', 'Autogluon_Tab', y_test, y_pred_iou_autg)


Test R2 score 0.4837
Test MAE score 0.1022
Test MSE score 0.0186
Test P95 score 0.2797
Test ECE score 0.0111


{'r2': 0.48366625422637777,
 'mae': 0.10221895102272285,
 'mse': 0.01855569656415807,
 'p95': 0.2796679973602294,
 'ece': 0.011104822744263646}

### LRP


#### Baseline

Predict Only the Mean


In [590]:
mean_lrp_train = np.mean(y_train_lrp)
lrp_pred_test = np.full_like(y_test_lrp, mean_lrp_train)
record_regression_results(model_results, 'LRP', 'Constant Mean Predictor', y_test_lrp, lrp_pred_test)


Test R2 score -0.0034
Test MAE score 0.1349
Test MSE score 0.0308
Test P95 score 0.3700
Test ECE score 0.0118


{'r2': -0.003390593920042262,
 'mae': 0.13488990933711548,
 'mse': 0.030807336125118975,
 'p95': 0.37001360592207166,
 'ece': 0.011791026592254727}

#### Linear Reg on Conf

In [591]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['mean_conf']),
    ],
    remainder='passthrough'
)
linear_reg_conf_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
linear_reg_conf_lrp_grid = GridSearchCV(
    linear_reg_conf_lrp_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_conf_lrp_grid.fit(conf_train, y_train_lrp)

best_idx = linear_reg_conf_lrp_grid.best_index_
mean_train_r2 = linear_reg_conf_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = linear_reg_conf_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -linear_reg_conf_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -linear_reg_conf_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -linear_reg_conf_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -linear_reg_conf_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {linear_reg_conf_lrp_grid.best_params_}")
best_linear_reg_conf_lrp = linear_reg_conf_lrp_grid.best_estimator_


Mean CV train r2 score 0.3108
Mean CV test r2 score 0.2297
Mean CV train MAE score 0.1123
Mean CV test MAE score 0.1135
Mean CV train MSE score 0.0218
Mean CV test MSE score 0.0223
Best params: {'model__fit_intercept': True, 'model__positive': False}


In [592]:
y_pred_lrp_baseline = best_linear_reg_conf_lrp.predict(conf_test)
record_regression_results(model_results, 'LRP', 'Univariate Linear Regression (Confidence)', y_test_lrp, y_pred_lrp_baseline)


Test R2 score 0.3028
Test MAE score 0.1115
Test MSE score 0.0214
Test P95 score 0.3295
Test ECE score 0.0308


{'r2': 0.3028491988868838,
 'mae': 0.11145128604366053,
 'mse': 0.021404784128860606,
 'p95': 0.3295279886411311,
 'ece': 0.03080790624660026}

#### Isotonic Regression on Conf


In [593]:
iso_lrp = IsotonicRegression(out_of_bounds="clip")
iso_lrp.fit(conf_train["mean_conf"].to_numpy(), y_train_lrp.to_numpy())
y_pred_lrp_iso = iso_lrp.predict(conf_test["mean_conf"].to_numpy())
record_regression_results(model_results, 'LRP', 'Isotonic Regression (Confidence)', y_test_lrp, y_pred_lrp_iso)


Test R2 score 0.4799
Test MAE score 0.0940
Test MSE score 0.0160
Test P95 score 0.2553
Test ECE score 0.0118


{'r2': 0.47986988867852975,
 'mae': 0.09395575650503854,
 'mse': 0.015969676480296947,
 'p95': 0.25525961079044274,
 'ece': 0.01182132664440277}

#### Linear Regression


Train models with cv and then test.


In [594]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
linear_reg_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
linear_reg_lrp_grid = GridSearchCV(
    linear_reg_lrp_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_lrp_grid.fit(X_train, y_train_lrp)

best_idx = linear_reg_lrp_grid.best_index_
mean_train_r2 = linear_reg_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = linear_reg_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -linear_reg_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -linear_reg_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -linear_reg_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -linear_reg_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {linear_reg_lrp_grid.best_params_}")
best_linear_reg_lrp = linear_reg_lrp_grid.best_estimator_


Mean CV train r2 score 0.4686
Mean CV test r2 score 0.4010
Mean CV train MAE score 0.0979
Mean CV test MAE score 0.0990
Mean CV train MSE score 0.0168
Mean CV test MSE score 0.0173
Best params: {'model__fit_intercept': True, 'model__positive': False}


In [595]:
y_pred_lrp_lr = best_linear_reg_lrp.predict(X_test)
record_regression_results(model_results, 'LRP', 'Linear Regression', y_test_lrp, y_pred_lrp_lr)


Test R2 score 0.4653
Test MAE score 0.0965
Test MSE score 0.0164
Test P95 score 0.2731
Test ECE score 0.0174


{'r2': 0.4652586132826322,
 'mae': 0.09653185397245294,
 'mse': 0.016418289886746686,
 'p95': 0.27314131938177444,
 'ece': 0.017403519229955776}

#### Decision Trees


In [596]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
decision_tree_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=SEED))
])
decision_tree_lrp_grid = GridSearchCV(
    decision_tree_lrp_pipeline,
    param_grid=TREE_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
decision_tree_lrp_grid.fit(X_train, y_train_lrp)

best_idx = decision_tree_lrp_grid.best_index_
mean_train_r2 = decision_tree_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = decision_tree_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -decision_tree_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -decision_tree_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -decision_tree_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -decision_tree_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {decision_tree_lrp_grid.best_params_}")
best_decision_tree_lrp = decision_tree_lrp_grid.best_estimator_


Mean CV train r2 score 0.5437
Mean CV test r2 score 0.4394
Mean CV train MAE score 0.0899
Mean CV test MAE score 0.0951
Mean CV train MSE score 0.0144
Mean CV test MSE score 0.0160
Best params: {'model__max_depth': None, 'model__max_features': None, 'model__min_samples_leaf': 100, 'model__min_samples_split': 100}


/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
180 fits failed out of a total of 360.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
180 fits failed with the following error:
Traceback (most recent call last):
  File "/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/felix/miniconda3/envs/pred_ad/lib/py

In [597]:
y_pred_lrp_dt = best_decision_tree_lrp.predict(X_test)
record_regression_results(model_results, 'LRP', 'Decision Tree', y_test_lrp, y_pred_lrp_dt)


Test R2 score 0.4936
Test MAE score 0.0932
Test MSE score 0.0155
Test P95 score 0.2509
Test ECE score 0.0110


{'r2': 0.49355645019658834,
 'mae': 0.09315785604803421,
 'mse': 0.015549454780353893,
 'p95': 0.2509189413124062,
 'ece': 0.0109895317168924}

#### Random Forest


In [598]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
rand_forest_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=SEED, n_jobs=-1))
])
rand_forest_lrp_grid = GridSearchCV(
    rand_forest_lrp_pipeline,
    param_grid=RF_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
rand_forest_lrp_grid.fit(X_train, y_train_lrp)

best_idx = rand_forest_lrp_grid.best_index_
mean_train_r2 = rand_forest_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = rand_forest_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -rand_forest_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -rand_forest_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -rand_forest_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -rand_forest_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {rand_forest_lrp_grid.best_params_}")
best_rand_forest_lrp = rand_forest_lrp_grid.best_estimator_


/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
360 fits failed out of a total of 720.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
360 fits failed with the following error:
Traceback (most recent call last):
  File "/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/felix/miniconda3/envs/pred_ad/lib/py

Mean CV train r2 score 0.6300
Mean CV test r2 score 0.4765
Mean CV train MAE score 0.0808
Mean CV test MAE score 0.0910
Mean CV train MSE score 0.0117
Mean CV test MSE score 0.0149
Best params: {'model__max_depth': None, 'model__max_features': None, 'model__min_samples_leaf': 2, 'model__min_samples_split': 100, 'model__n_estimators': 200}


In [599]:
y_pred_lrp_rf = best_rand_forest_lrp.predict(X_test)
record_regression_results(model_results, 'LRP', 'Random Forest', y_test_lrp, y_pred_lrp_rf)


Test R2 score 0.5337
Test MAE score 0.0888
Test MSE score 0.0143
Test P95 score 0.2449
Test ECE score 0.0151


{'r2': 0.5336644167697187,
 'mae': 0.08878773034487171,
 'mse': 0.014318010500329154,
 'p95': 0.24491219423203237,
 'ece': 0.015131459725718619}

#### MLP


In [600]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
mlp_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', MLPRegressor(max_iter=500, random_state=SEED))
])
mlp_lrp_grid = GridSearchCV(
    mlp_lrp_pipeline,
    param_grid=MLP_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
mlp_lrp_grid.fit(X_train, y_train_lrp)

best_idx = mlp_lrp_grid.best_index_
mean_train_r2 = mlp_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = mlp_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -mlp_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -mlp_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -mlp_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -mlp_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {mlp_lrp_grid.best_params_}")
best_mlp_lrp = mlp_lrp_grid.best_estimator_


Mean CV train r2 score 0.5411
Mean CV test r2 score 0.4339
Mean CV train MAE score 0.0908
Mean CV test MAE score 0.0959
Mean CV train MSE score 0.0145
Mean CV test MSE score 0.0163
Best params: {'model__activation': 'tanh', 'model__alpha': 0.01, 'model__hidden_layer_sizes': (100, 50), 'model__learning_rate_init': 0.01}


In [601]:
y_pred_lrp_mlp = best_mlp_lrp.predict(X_test)
record_regression_results(model_results, 'LRP', 'MLP', y_test_lrp, y_pred_lrp_mlp)


Test R2 score 0.5013
Test MAE score 0.0942
Test MSE score 0.0153
Test P95 score 0.2489
Test ECE score 0.0321


{'r2': 0.5013390038911385,
 'mae': 0.09418657675461162,
 'mse': 0.015310505213721922,
 'p95': 0.24890750903150932,
 'ece': 0.03210412275684554}

#### XGBoost

In [602]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
xgb_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb.XGBRegressor())
])
xgb_lrp_grid = GridSearchCV(
    xgb_lrp_pipeline,
    param_grid=XGB_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
xgb_lrp_grid.fit(X_train, y_train_lrp)


best_idx = xgb_lrp_grid.best_index_
mean_train_r2 = xgb_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = xgb_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -xgb_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -xgb_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -xgb_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -xgb_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {xgb_lrp_grid.best_params_}")
best_xgb_lrp = xgb_lrp_grid.best_estimator_

Mean CV train r2 score 0.7827
Mean CV test r2 score 0.4581
Mean CV train MAE score 0.0645
Mean CV test MAE score 0.0926
Mean CV train MSE score 0.0069
Mean CV test MSE score 0.0155
Best params: {'model__learning_rate': 0.1, 'model__n_estimators': 50}


In [603]:
y_pred_lrp_xgb = best_xgb_lrp.predict(X_test)
record_regression_results(model_results, 'LRP', 'XGBoost', y_test_lrp, y_pred_lrp_xgb)


Test R2 score 0.5338
Test MAE score 0.0885
Test MSE score 0.0143
Test P95 score 0.2471
Test ECE score 0.0122


{'r2': 0.5337690392429528,
 'mae': 0.08850661801320162,
 'mse': 0.01431479825205944,
 'p95': 0.24713039398193354,
 'ece': 0.012185188428428383}

#### Autogluon

##### Tabluar

In [604]:
train_lrp = pd.merge(X_train, y_train_lrp, left_index=True, right_index=True)

In [605]:
autoglue_model_lrp = TabularPredictor(label="lrp", problem_type="regression", eval_metric="r2", path="../models/assessors/autoglue_tab/lrp/tab/")
autoglue_predictor_lrp = autoglue_model_lrp.fit(train_lrp)


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.14
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #100-Ubuntu SMP PREEMPT_DYNAMIC Tue Jan 13 16:40:06 UTC 2026
CPU Count:          32
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 23.55/23.55 GB
Total GPU Memory:   Free: 23.55 GB, Allocated: 0.00 GB, Total: 23.55 GB
GPU Count:          1
Memory Avail:       3.98 GB / 30.46 GB (13.1%)
Disk Space Avail:   1586.98 GB / 3542.28 GB (44.8%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : New in v1.5: The state-of-the-art for tabular data. Massively better than 'best' on datasets <100000 samples by using new 

In [606]:
autoglue_predictor_lrp.fit_summary(verbosity=2)

*** Summary of fit() ***
Estimated performance of each model:
                 model  score_val eval_metric  pred_time_val  fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0  WeightedEnsemble_L2   0.568838          r2       0.044627  4.941714                0.000257           0.015438            2       True          9
1              XGBoost   0.561056          r2       0.011494  0.388315                0.011494           0.388315            1       True          6
2             CatBoost   0.558967          r2       0.001415  0.787329                0.001415           0.787329            1       True          4
3           LightGBMXT   0.557806          r2       0.001401  0.542780                0.001401           0.542780            1       True          1
4             LightGBM   0.554426          r2       0.001218  0.400250                0.001218           0.400250            1       True          2
5        ExtraTreesMSE   0.550784          r

/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/autogluon/core/utils/plots.py:169: UserWarning: AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"
  warnings.warn('AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"')


{'model_types': {'LightGBMXT': 'LGBModel',
  'LightGBM': 'LGBModel',
  'RandomForestMSE': 'RFModel',
  'CatBoost': 'CatBoostModel',
  'ExtraTreesMSE': 'XTModel',
  'XGBoost': 'XGBoostModel',
  'NeuralNetTorch': 'TabularNeuralNetTorchModel',
  'LightGBMLarge': 'LGBModel',
  'WeightedEnsemble_L2': 'WeightedEnsembleModel'},
 'model_performance': {'LightGBMXT': 0.5578059292703965,
  'LightGBM': 0.5544262821016888,
  'RandomForestMSE': 0.5461038384847465,
  'CatBoost': 0.5589666463068765,
  'ExtraTreesMSE': 0.5507839898625817,
  'XGBoost': 0.5610564675564234,
  'NeuralNetTorch': 0.5366989885647722,
  'LightGBMLarge': 0.5336999636151871,
  'WeightedEnsemble_L2': 0.5688381263911423},
 'model_best': 'WeightedEnsemble_L2',
 'model_paths': {'LightGBMXT': ['LightGBMXT'],
  'LightGBM': ['LightGBM'],
  'RandomForestMSE': ['RandomForestMSE'],
  'CatBoost': ['CatBoost'],
  'ExtraTreesMSE': ['ExtraTreesMSE'],
  'XGBoost': ['XGBoost'],
  'NeuralNetTorch': ['NeuralNetTorch'],
  'LightGBMLarge': ['LightG

In [607]:
y_pred_lrp_autg = autoglue_predictor_lrp.predict(X_test)
record_regression_results(model_results, 'LRP', 'Autogluon_Tab', y_test_lrp, y_pred_lrp_autg)


Test R2 score 0.5446
Test MAE score 0.0877
Test MSE score 0.0140
Test P95 score 0.2412
Test ECE score 0.0119


{'r2': 0.5446497834256714,
 'mae': 0.0877430853418219,
 'mse': 0.013980724218119321,
 'p95': 0.24124779105186447,
 'ece': 0.011863464418171071}

### Save predictions

In [612]:
test_inst = X_test.copy()
print(f"Number of test instances: {test_inst.shape[0]}")
metrics = ["iou", "lrp"]
models = ["baseline", "lr", "dt", "rf", "mlp", "xgb", "autg"]
for metric in metrics:
    for model in models:
        if metric == "iou":
            test_inst["GT"] = y_test
        else:
            test_inst["GT"] = y_test_lrp
        var_name = f"y_pred_{metric}_{model}"
        test_inst[model] = globals()[var_name]
    test_inst.to_csv(f"../results/assessors/{metric}_test_ass_" + data_path_file)


Number of test instances: 4137


# Final Model Comparison


In [613]:

results_df = pd.DataFrame(model_results, index=None)
results_df["test_r2"] = results_df["test_r2"].round(4)
results_df["test_mae"] = results_df["test_mae"].round(4)
#results_df["test_mse"] = results_df["test_mse"].round(4)
results_df.to_csv(f"../results/assessors/{model_det}_ass_results_table.csv")
display(results_df)


,target,model,test_r2,test_mae,test_ece
0,IoU,Constant Mean Predictor,-0.0063,0.1472,0.013445
1,IoU,Univariate Linear Regression (Confidence),0.3245,0.1214,0.039087
2,IoU,Isotonic Regression (Confidence),0.4291,0.1083,0.009343
3,IoU,Linear Regression,0.4462,0.1078,0.016595
4,IoU,Decision Tree,0.4417,0.1069,0.017860
5,IoU,Random Forest,0.4759,0.1033,0.013352
6,IoU,MLP,0.4679,0.1039,0.010388
7,IoU,XGBoost,0.4664,0.1038,0.011631
8,IoU,Autogluon_Tab,0.4837,0.1022,0.011105
9,LRP,Constant Mean Predictor,-0.0034,0.1349,0.011791


In [614]:
print(results_df.to_latex(index=False))

\begin{tabular}{llrrr}
\toprule
target & model & test_r2 & test_mae & test_ece \\
\midrule
IoU & Constant Mean Predictor & -0.006300 & 0.147200 & 0.013445 \\
IoU & Univariate Linear Regression (Confidence) & 0.324500 & 0.121400 & 0.039087 \\
IoU & Isotonic Regression (Confidence) & 0.429100 & 0.108300 & 0.009343 \\
IoU & Linear Regression & 0.446200 & 0.107800 & 0.016595 \\
IoU & Decision Tree & 0.441700 & 0.106900 & 0.017860 \\
IoU & Random Forest & 0.475900 & 0.103300 & 0.013352 \\
IoU & MLP & 0.467900 & 0.103900 & 0.010388 \\
IoU & XGBoost & 0.466400 & 0.103800 & 0.011631 \\
IoU & Autogluon_Tab & 0.483700 & 0.102200 & 0.011105 \\
LRP & Constant Mean Predictor & -0.003400 & 0.134900 & 0.011791 \\
LRP & Univariate Linear Regression (Confidence) & 0.302800 & 0.111500 & 0.030808 \\
LRP & Isotonic Regression (Confidence) & 0.479900 & 0.094000 & 0.011821 \\
LRP & Linear Regression & 0.465300 & 0.096500 & 0.017404 \\
LRP & Decision Tree & 0.493600 & 0.093200 & 0.010990 \\
LRP & Random Fore